# Hyperparameter Tuning: K & Threshold (of the retriever)

Grid search over K (number of neighbors) and threshold (minimum cosine similarity). Trained on IHC with the full index. Only the best (K, threshold) per model is saved.

## 1. Imports

In [1]:
import os
import json
import shutil
from itertools import product
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import faiss
from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
from sklearn.metrics import f1_score, precision_score, recall_score, classification_report
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

import sys
sys.path.insert(0, str(Path("..").resolve()))
from retriever import retrieve_top_k_above_threshold

## 2. Configuration

Edit `K_VALUES` and `THRESHOLD_VALUES` to define the search grid.

In [2]:
# Define paths
WEIGHTS_RAC_DIR = Path('../..') / 'weigths' / 'weights_rac_hyperparameter_tuning'
INDEX_DIR       = Path('../..') / 'corpus' / 'index'
CHUNKS_DIR      = Path('../..') / 'corpus' / 'chunks'

# Models used
MODELS = {
    'bert':     'bert-base-uncased',
    'hatebert': 'GroNLP/hateBERT',
    'roberta':  'roberta-base',
}

# Contrastive retriever — shared across all configs
RETRIEVER_HF_ID = 'sentence-transformers/all-mpnet-base-v2'

# What to run
SELECTED_MODELS = ['bert', 'hatebert', 'roberta']
INDEX_TYPE      = 'full'
DATASET         = 'IHC'

# Hyperparameter grid
K_VALUES         = [3, 5, 10]
THRESHOLD_VALUES = [0.3, 0.4, 0.5, 0.6]
MAX_K            = max(K_VALUES)
MIN_THRESHOLD    = min(THRESHOLD_VALUES)

# Training config
MAX_LENGTH    = 256
BATCH_SIZE    = 16
LEARNING_RATE = 2e-5
NUM_EPOCHS    = 3

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')


Device: cuda


## 3. Load Dataset

Load IHC using `data_loaders.py`.

In [3]:
from data_loaders import load_ihc_binary

# Load IHC
train_ihc, test_ihc = load_ihc_binary(seed=42)

print(f'IHC — train: {len(train_ihc):,}  test: {len(test_ihc):,}')


README.md:   0%|          | 0.00/792 [00:00<?, ?B/s]

implicit_hate_v1_stg1_posts.tsv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/21480 [00:00<?, ? examples/s]

Map:   0%|          | 0/19332 [00:00<?, ? examples/s]

Map:   0%|          | 0/2148 [00:00<?, ? examples/s]

IHC — train: 19,332  test: 2,148


## 4. Self-Exclusion Lookup

`chunks_example.csv` maps raw tweet text → `chunk_id` in the FAISS index.
Used at train time to pass `chunk_id` so a model never retrieves itself as a neighbor.
The full example file (including IHC) is used here because the full index contains IHC tweets — using the noihc variant would leave IHC tweets with no chunk_id, causing them to silently retrieve themselves (data leakage).

In [4]:
from training_utils import filter_records, strip_label_prefix, set_seed

chunks_df = pd.read_csv(CHUNKS_DIR / 'chunks_example.csv')

text_to_chunk_id = {
    strip_label_prefix(row.text): int(row.chunk_id)
    for _, row in chunks_df.iterrows()
}
print(f'Self-exclusion lookup: {len(text_to_chunk_id):,} entries')

Self-exclusion lookup: 67,864 entries


## 5. Augmentation Functions

Augment once at MAX_K / MIN_THRESHOLD and cache the results. `filter_records` then applies the actual (K, threshold) per combo without re-encoding.

In [5]:
def augment_split_cached(hf_dataset, text_col, is_train, ret_model, ret_tokenizer, ret_index, ret_documents):
    """Retrieve up to MAX_K neighbors above MIN_THRESHOLD, storing (text, score) pairs for later filtering."""
    records = []
    for example in tqdm(hf_dataset, desc=f"{'train' if is_train else 'test'}"):
        tweet    = example[text_col]
        chunk_id = text_to_chunk_id.get(tweet) if is_train else None
        neighbors = retrieve_top_k_above_threshold(
            tweet, MIN_THRESHOLD, ret_model, ret_tokenizer, ret_index, ret_documents,
            chunk_id=chunk_id, k=MAX_K, use_mean_pool=True,
        )
        records.append({
            'query':     tweet,
            'neighbors': neighbors,   # list of (text, score)
            'label':     example['label'],
        })
    return records

## 6. Tokenization

Assembles the augmented string using the model's `sep_token` then tokenizes.

In [6]:
from training_utils import tokenize_augmented

## 7. Metrics

In [7]:
from training_utils import compute_metrics

## 8. Hyperparameter Tuning Loop

Sweep all (K, threshold) combos per model. Only the best combo's weights are saved.

In [ ]:
results = {}

# Load the sbert retriever
print(f"Loading retriever: {RETRIEVER_HF_ID} ...")
ret_tokenizer = AutoTokenizer.from_pretrained(RETRIEVER_HF_ID)
ret_model     = AutoModel.from_pretrained(RETRIEVER_HF_ID).eval().to(device)
print(f"Retriever ready on {device}\n")

# Load the full FAISS index
index_path = INDEX_DIR / f'vdb_{INDEX_TYPE}.faiss'
ret_index  = faiss.read_index(str(index_path))
with open(INDEX_DIR / f'lookup_{INDEX_TYPE}.json') as f:
    ret_documents = json.load(f)
print(f"Index: {INDEX_TYPE}  |  Vectors: {ret_index.ntotal:,}\n")

# Augment IHC train and test once at MAX_K / MIN_THRESHOLD
print(f"Augmenting IHC train (MAX_K={MAX_K}, MIN_THRESHOLD={MIN_THRESHOLD}) ...")
cached_train = augment_split_cached(train_ihc, 'post', True,
                                    ret_model, ret_tokenizer, ret_index, ret_documents)
print(f"Augmenting IHC test ...")
cached_test  = augment_split_cached(test_ihc,  'post', False,
                                    ret_model, ret_tokenizer, ret_index, ret_documents)

del ret_model, ret_tokenizer
if device.type == 'cuda':
    torch.cuda.empty_cache()
print("Augmentation done. Retriever freed.\n")

# HP tuning loop — sweep all (K, threshold) combos per model
for model_name, hf_id in MODELS.items():
    if model_name not in SELECTED_MODELS:
        continue

    best_f1        = 0.0
    best_k         = None
    best_threshold = None

    tokenizer = AutoTokenizer.from_pretrained(hf_id)

    for k, threshold in product(K_VALUES, THRESHOLD_VALUES):
        print(f"  {model_name} | K={k} | t={threshold}", end=' ', flush=True)

        tok_train = tokenize_augmented(filter_records(cached_train, k, threshold), tokenizer)
        tok_test  = tokenize_augmented(filter_records(cached_test,  k, threshold), tokenizer)

        # Seed before model init so every config starts with identical classifier head weights.
        # TrainingArguments(seed=42) only seeds the training loop, not from_pretrained().
        set_seed(42)
        model = AutoModelForSequenceClassification.from_pretrained(hf_id, num_labels=2)

        training_args = TrainingArguments(
            output_dir=str(Path('../..') / 'checkpoints_hp_tuning' / model_name / f'k{k}_t{threshold}'),
            num_train_epochs=NUM_EPOCHS,
            per_device_train_batch_size=BATCH_SIZE,
            per_device_eval_batch_size=BATCH_SIZE * 2,
            learning_rate=LEARNING_RATE,
            eval_strategy='epoch',
            save_strategy='no',
            logging_strategy='epoch',
            report_to='none',
            seed=42,
        )

        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=tok_train,
            eval_dataset=tok_test,
            compute_metrics=compute_metrics,
        )

        trainer.train()

        preds_out = trainer.predict(tok_test)
        preds  = np.argmax(preds_out.predictions, axis=-1)
        labels = [r['label'] for r in filter_records(cached_test, k, threshold)]
        macro_f1 = f1_score(labels, preds, average='macro', zero_division=0)

        print(classification_report(labels, preds, target_names=['Non-HS', 'HS']))

        results[(model_name, k, threshold)] = {
            'macro_f1': macro_f1,
            'macro_p':  precision_score(labels, preds, average='macro', zero_division=0),
            'macro_r':  recall_score(labels, preds, average='macro',    zero_division=0),
        }

        if macro_f1 > best_f1:
            best_f1        = macro_f1
            best_k         = k
            best_threshold = threshold

        del model
        if device.type == 'cuda':
            torch.cuda.empty_cache()

    print(f"\n>>> Best for {model_name}: K={best_k}, threshold={best_threshold}, F1={best_f1:.3f}")

## 9. Results

Full grid showing macro F1/P/R for every (model, K, threshold) combination. Best F1 per model highlighted.

In [9]:
# Build and display the results table
rows = {}
for (model_name, k, threshold), vals in results.items():
    row_key = f"{model_name} | K={k} | t={threshold}"
    rows[row_key] = {
        'Model': model_name,
        'F1':        vals['macro_f1'],
        'Precision': vals['macro_p'],
        'Recall':    vals['macro_r'],
    }

df = pd.DataFrame(rows).T
df[['F1', 'Precision', 'Recall']] = df[['F1', 'Precision', 'Recall']].astype(float)

# Highlight best F1 per model
def highlight_best(s):
    styles = [''] * len(s)
    for model_name in SELECTED_MODELS:
        mask = df['Model'] == model_name
        if mask.any():
            best_idx = df.loc[mask, 'F1'].idxmax()
            styles[df.index.get_loc(best_idx)] = 'font-weight: bold; background-color: #d4f1d4'
    return styles

styled = (
    df[['F1', 'Precision', 'Recall']]
    .style
    .format('{:.3f}')
    .apply(highlight_best, axis=0)
    .set_caption('HP Tuning — IHC, full index (best per model highlighted)')
)
display(styled)

,F1,Precision,Recall
bert | K=3 | t=0.3,0.777,0.779,0.775
bert | K=3 | t=0.4,0.779,0.782,0.777
bert | K=3 | t=0.5,0.787,0.796,0.782
bert | K=3 | t=0.6,0.785,0.791,0.780
bert | K=5 | t=0.3,0.775,0.783,0.771
bert | K=5 | t=0.4,0.775,0.780,0.772
bert | K=5 | t=0.5,0.778,0.785,0.773
bert | K=5 | t=0.6,0.779,0.784,0.775
bert | K=10 | t=0.3,0.780,0.786,0.776
bert | K=10 | t=0.4,0.776,0.782,0.772
